# 🎓 Axelion-LLM: Dedicated GPU Training Notebook
### Sign in via College / Institutional ID • No GitHub Required • Direct Drive Sync

---

### 📌 How to Use with your College ID:
1. **Log in with College ID**: In the top right corner of Google Colab, make sure you are signed in with your **College / University Google account** (e.g. `yourname@college.edu`).
2. **Enable GPU**: Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
3. **Run cells in order**: All training steps, dataset tokenization, and checkpoint saving to your College Google Drive are automated below without needing GitHub.

## 1. ⚡ Verify Hardware & GPU Acceleration
Check that Google Colab has allocated a GPU for your session.

In [ ]:
!nvidia-smi

import torch
print("="*50)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model       : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU not detected! Go to Runtime -> Change runtime type and select T4 GPU.")
print("="*50)

## 2. 🔑 Connect to Google Drive (Sign in with College ID)
When running the cell below, a popup will ask for permission. **Select your College / University account** to mount your College Google Drive storage for saving models and reading data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Successfully connected to College Google Drive!")

## 3. 📂 Load Project Files (No GitHub Needed)
Choose **either Method A or Method B** to load your `Axelion-LLM` project files into Colab:

In [ ]:
import os
import zipfile

# Check if Axelion-LLM is already cloned or uploaded
if os.path.exists("/content/Axelion-LLM"):
    %cd /content/Axelion-LLM
elif os.path.exists("/content/drive/MyDrive/Axelion-LLM"):
    print("Loading project from College Google Drive...")
    %cd /content/drive/MyDrive/Axelion-LLM
elif os.path.exists("/content/Axelion-LLM.zip"):
    print("Unzipping Axelion-LLM.zip...")
    with zipfile.ZipFile("/content/Axelion-LLM.zip", "r") as zip_ref:
        zip_ref.extractall("/content/")
    %cd /content/Axelion-LLM
else:
    print("Cloning from GitHub repository...")
    !git clone https://github.com/Akshattermux/Axelion-LLM.git /content/Axelion-LLM
    %cd /content/Axelion-LLM

!pwd
!ls -la


## 4. 🛠️ Install Training Dependencies

In [ ]:
!pip install -q tiktoken datasets transformers torch tqdm numpy
print("✅ All dependencies installed!")

## 5. 📊 Prepare & Tokenize Training Data
This step processes your text data and prepares the binary token files (`train.bin`, `validation.bin`) and instruction tuning datasets.

In [ ]:
!python scripts/prepare_data.py --mode all

print("\n--- Pretraining Data ---")
!ls -lh data/pretraining/
print("\n--- Instruction Fine-Tuning Data ---")
!ls -lh data/instruction/

## 6. 🏋️ Pretrain Axelion-LLM on GPU

Choose your configuration:
- **size**: `micro` (Fastest, ~5M), `mini` (Balanced, ~15M), `medium` (~40M), or `large` (~100M)
- **batch-size**: Micro batch size per GPU forward pass
- **grad-accum**: Gradient accumulation steps
- **max-iters**: Total training steps
- **lr**: Peak learning rate (default: `3e-4`)

In [ ]:
# Run Pretraining on Colab GPU (1B Model)
!python train.py \
    --size 1b \
    --batch-size 1 \
    --grad-accum 16 \
    --max-iters 2000 \
    --eval-interval 200 \
    --lr 3e-4 \
    --checkpoint-out checkpoints/axelion-base-1b.pt \
    --device cuda


## 7. 🎯 Supervised Fine-Tuning (SFT / Chat Alignment)
Fine-tune Axelion on conversational instructions using prompt loss masking (loss is computed only on assistant tokens).

In [ ]:
# Run Instruction Fine-Tuning (SFT)
!python training/sft.py \
    --base-ckpt checkpoints/axelion-base-1b.pt \
    --output-ckpt checkpoints/axelion-instruct-1b.pt \
    --epochs 3 \
    --batch-size 1 \
    --grad-accum 8 \
    --lr 1e-4 \
    --device cuda


## 8. 📈 Model Evaluation (Loss & Perplexity)
Measure test validation loss and perplexity on the newly trained weights.

In [ ]:
!python training/evaluate.py \
    --checkpoint checkpoints/axelion-instruct-1b.pt \
    --device cuda


## 9. 💾 Backup Checkpoints to College Google Drive
Save your trained weights permanently into your College Google Drive so they are never lost when Colab disconnects.

In [ ]:
import os

drive_dest = "/content/drive/MyDrive/Axelion_Checkpoints"
!mkdir -p "{drive_dest}"
!cp -v checkpoints/* "{drive_dest}/"
print(f"\n✅ Checkpoints safely backed up to College Drive at: {drive_dest}")
!ls -lh "{drive_dest}"

## 10. 💬 Test Generation & Chat with Your Model
Test prompts with the trained model using fast KV-caching.

In [ ]:
import os
import torch
from axelion.config import ModelConfig
from axelion.model import Axelion
from axelion.tokenizer import AxelionTokenizer
from axelion.generation import stream_generate

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AxelionTokenizer()

# Pick trained checkpoint
candidates = [
    "checkpoints/axelion-instruct-1b.pt",
    "checkpoints/axelion-base-1b.pt",
    "/content/drive/MyDrive/Axelion_Checkpoints/axelion-instruct-1b.pt",
    "checkpoints/axelion-instruct-mini.pt",
    "checkpoints/axelion-base-mini.pt",
    "ckpt.pt"
]
ckpt_path = next((c for c in candidates if os.path.exists(c)), None)

if ckpt_path:
    print(f"Loading: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    config = ckpt.get("config", ModelConfig.create_1b(vocab_size=tokenizer.vocab_size))
    config.vocab_size = max(config.vocab_size, tokenizer.vocab_size)
    model = Axelion(config).to(device)
    model.load_checkpoint_weights(ckpt["model"])
else:
    print("Using initialized 1B architecture configuration.")
    config = ModelConfig.create_1b(vocab_size=tokenizer.vocab_size)
    model = Axelion(config).to(device)

model.eval()

# Type your prompt here:
user_prompt = "Explain how transformer self-attention works:"
print(f"\nPrompt: {user_prompt}\n" + "="*50)

for chunk in stream_generate(
    model=model,
    tokenizer=tokenizer,
    prompt=user_prompt,
    max_new_tokens=150,
    temperature=0.7,
    top_p=0.9,
    stop_token_ids=[tokenizer.end_id, tokenizer.pad_id],
    device=device
):
    print(chunk, end="", flush=True)
print("\n" + "="*50)
